# 09 — Excel ingestion

Most real-world digital-twin projects don't start with a hand-written TTL — they start with a populated **spreadsheet** of components and attributes. The Replica Builder's *Excel Import* tab takes a workbook in the standard 6-row-header template and emits a single TTL blob ready for upload to GraphDB.

This notebook walks through that path end-to-end:

1. The 6-row header convention (what each row means).
2. The supported attribute types and which header cells each one reads.
3. Running the converter on the bundled `alpine_village_replica_template.xlsx`.
4. Validating the output and uploading to a fresh named graph.
5. Where the same flow lives in the Streamlit UI.

Prerequisites: complete `00_setup.md` and run `01_ontology_basics.ipynb` at least once.

## 9.1 Setup

In [ ]:
import os, sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")
# Fuseki needs admin auth for writes (upload / SPARQL update). Compose defaults:
os.environ.setdefault("FUSEKI_ADMIN_USER", "admin")
os.environ.setdefault("FUSEKI_ADMIN_PASSWORD", "admin")

from backend.graphdb import GraphDBClient
from backend.replica_builder.utils.create_class_and_attribute_graph import process_excel_to_ttl

client = GraphDBClient(token="local", selected_repo="workspace_demo")

TEMPLATE_PATH = pathlib.Path('sample_data/alpine_village_replica_template.xlsx').resolve()
EXCEL_GRAPH   = "<https://digicities.info/tutorial/alpine_village_excel>"
PROJECT_URI   = "https://digicities.info/tutorial/alpine_village"
OUTPUT_TTL    = pathlib.Path('alpine_village_excel.ttl').resolve()

TEMPLATE_PATH.exists(), client.auth_mode

## 9.2 Inspect the template

The template is just a regular .xlsx workbook — open it in Excel/LibreOffice if you want to see the layout. From Python, `pandas.read_excel` with `header=[0,1,2,3,4,5]` reads the six header rows as a `MultiIndex` over columns. We do the same here just to see what the importer will see.

In [ ]:
import pandas as pd

sheets = pd.read_excel(TEMPLATE_PATH, sheet_name=None, header=[0, 1, 2, 3, 4, 5])
for name, df in sheets.items():
    print(f'{name:18s}  {len(df):>3} rows × {len(df.columns):>2} cols')

Each column is a tuple of one entry per header row (6 by default, 7 in `LinkedClassObjectType` mode — see §9.3). Here are the columns of the `EnergyConsumer` sheet — note the empty cells (which pandas labels `Unnamed: …_level_N` and the importer filters out):

In [ ]:
for col in sheets['EnergyConsumer'].columns:
    print(col)

## 9.3 Header rows — what each one means

The importer interprets the header rows positionally. By default it reads **six** rows; if any sheet has the literal string `LinkedClassObjectType` somewhere in row 7, the converter switches into **seven**-row mode for the whole workbook. Every column always has all the header rows in use; cells the column doesn't need are simply blank.

| Row | Purpose                       | Used by                                                                    |
|-----|-------------------------------|----------------------------------------------------------------------------|
| 1   | **Attribute name**            | All. The column named `id` marks the instance ID column for the sheet.    |
| 2   | **Attribute type**            | All. One of `Physical`, `UnitBasedCost`, `SimpleCost`, `Categorical`, `Event`, `ClassObject`, `Identifier`, `Annotation`, `Curve`, `CustomPhysicalRatio`, `Resource`, `SimpleValue`, `Historic`, `Live`, `Future`. |
| 3   | **Unit (or x-axis unit)**     | `Physical`, `UnitBasedCost`, `Curve` (x-axis), `CustomPhysicalRatio` (numerator), `Historic`/`Live`/`Future`. QUDT short code (`KiloW`, `M2`, `KiloW-HR`, `PERCENT`). |
| 4   | **y-axis / denominator unit** | `Curve` (y-axis), `CustomPhysicalRatio` (denominator).                     |
| 5   | **Currency**                  | `SimpleCost`, `UnitBasedCost`. ISO code (`CHF`, `EUR`, `USD`).             |
| 6   | **Predicate**                 | `ClassObject`. The `dici_onto:` property name connecting the instance to the target (e.g. `locatedIn`, `installedAt`). Use the bare property name, no prefix. |
| 7   | **LinkedClassObjectType** — *optional*  | `ClassObject` only. A full URI prefix (must end with `/` or `#`) that gets prepended to the cell value, **overriding** the converter's `uri_mode`. To enable this row, put the literal string `LinkedClassObjectType` in the `id` column — the importer scans for that marker to decide whether to read 6 or 7 header rows. If you turn it on, every sheet must have a row 7 (even if just the marker), or pandas will eat the first instance row as a phantom header level. |

Each spreadsheet **sheet** corresponds to a `dici_onto:` class — the sheet name becomes the `rdf:type` of every instance row in it. The bundled template uses `EnergyConsumer`, `EnergyConverter`, `Network`, and `Location`. You can add more sheets for any class your ontology defines.

Two sheet names are reserved: `Reference` (citation list, see §9.6) and `Data Validation` (Excel dropdown helper, ignored by the importer).

## 9.4 Attribute type cheatsheet

Each row in the table below is one of the rows we just printed. The 'Reads cells' column tells you which header rows the importer reads for that type — anything else can stay blank.

| Type                   | Cell value     | Reads cells              | Output snippet                                                                            |
|------------------------|----------------|--------------------------|-------------------------------------------------------------------------------------------|
| `Physical`             | number         | name, type, unit         | `qudt:value 4800.0 ; qudt:unit unit:KiloW-HR`                                            |
| `SimpleCost`           | number         | name, type, currency     | `qudt:value 250.0 ; dici_onto:currency cur:CHF`                                          |
| `UnitBasedCost`        | number         | name, type, unit, currency | adds a `qudt:unit` triple on top of `SimpleCost`                                       |
| `Categorical`          | category name  | name, type               | `dici_onto:hasCategoricalValue dici_onto:SingleFamilyHouse`                              |
| `Event`                | year/date/datetime | name, type           | `dici_onto:hasTemporalValue "1970"^^xsd:gYear` (precision auto-detected)                  |
| `ClassObject`          | target id      | name, type, predicate, *(optional)* LinkedClassObjectType | `dici_onto:locatedIn <…/Location/AlpineValley>`                                          |
| `Identifier`           | string         | name, type               | `dici_onto:hasIdentifier <…/buildingId>` plus `dici_onto:identifierValue "BLDG-A-001"`     |
| `Annotation`           | string         | name, type               | `rdfs:<colname>` if `<colname>` is one of the four valid W3C-defined RDFS annotation properties (`label`, `comment`, `seeAlso`, `isDefinedBy`); otherwise `:<colname>` (project-scoped). **Don't use Annotation for closed-vocabulary values** (e.g. `CarrierCategory`, `BaseCarrier`) — use `Categorical` instead so values get linked to ontology individuals rather than stored as free-form strings. |
| `Curve`                | `[(x,y);(x,y);…]` | name, type, unit (x), unit_y (y) | `dici_onto:hasDataPoints """[ … ]"""`                                          |
| `CustomPhysicalRatio`  | number         | name, type, unit, unit_y | `qudt:value 0.25 ; dici_onto:hasUnitLabel "CHF/KiloW-HR"`                                 |
| `Historic`/`Live`/`Future` | path or URL | name, type, unit       | creates a `dici_onto:TimeSeries` node and links it via `dici_onto:hasHistoricTimeSeries` |
| `Resource`             | path           | name, type               | `dici_onto:hasDataPath "…"`                                                              |
| `SimpleValue`          | string/number  | name, type               | `dici_onto:hasAttributeValue …` (no unit, no currency)                                   |

Add `<name>_datasource` as a sibling column (with no other header rows filled) to attach a citation. The cell value matches an `id` from the `Reference` sheet — emits `prov:wasDerivedFrom <…/Reference/<id>>`. Anything that doesn't match becomes a free-form `dcterms:source` string.

**LinkedClassObjectType** lets you point a `ClassObject` attribute at instances in *another* namespace. Put the literal string `LinkedClassObjectType` in the `id` column's row 7 (this is the marker that switches the importer into 7-row mode for the whole workbook), then on each `ClassObject` column put the desired URI prefix — e.g. `https://example.org/locations/`. Cell values then become bare instance IDs (`AlpineValley` → `<https://example.org/locations/AlpineValley>`), bypassing whatever `uri_mode` the converter is called with.

### When to use Annotation vs Categorical vs ClassObject

| You're storing… | Use this type | Why |
|---|---|---|
| A free-form human-readable label or comment | `Annotation` with column name `label` or `comment` | Emits `rdfs:label` / `rdfs:comment` — semantically meaningful to every RDF tool |
| A value from a closed vocabulary (one of N options) | `Categorical` | Cell value becomes an ontology individual, queryable + comparable across rows |
| A pointer to another instance in the same project | `ClassObject` | Creates a typed predicate edge (e.g. `dici_onto:locatedIn`) between instances |
| A pointer to an instance in *another* namespace | `ClassObject` + `LinkedClassObjectType` in row 7 | Same as above, but the target IRI is built from a foreign namespace prefix |
| A bare project-scoped string with no semantics intended | `Annotation` with any other column name | Emits `:<colname> "…"` under the project namespace — well-formed but means nothing outside this project |
| A number, identifier, time, curve, or time-series | the corresponding typed attribute (`Physical`, `Identifier`, `Event`, `Curve`, `Historic`, etc.) | Each one carries its proper semantics + unit + provenance |

## 9.4b Per-attribute-type CSV reference

If you'd rather copy values into a validated workbook than re-derive every example from the cheatsheet, the repo ships a CSV reference under [`data/ingestion_template/`](../data/ingestion_template/):

- `attribute_types.csv` — every supported attribute type as a column with one demo row, plus three rows showing year/date/datetime variants of `Event`. Layout matches the 7-row header convention.
- `reference.csv` — the citation-sheet equivalent.
- `README.md` — the workflow (paste values only into your validated `.xlsx`, leave Excel data-validation rules intact).

These ship in CSV (not `.xlsx`) precisely so you can paste cell values into your own template without disturbing its data-validation dropdowns.

## 9.5 Run the conversion

`process_excel_to_ttl` takes a project URI, the path to your .xlsx, the output path, and a URI mode. We use `default` here — that means instance URIs are constructed as `<{project_uri}/{sheet_name}/{row_id}>`. Two other modes exist for cases where you want full control over the URI shape:

- `default` — `<{project_uri}/{sheet_name}/{row_id}>`. Pick this unless you have a reason not to.
- `complete-project-uri` — `<{project_uri}{row_id}>` (no separator inserted, project URI must end with `/` or `#`).
- `full-uri-in-cell` — the `id` cell already contains the full URI; the importer uses it verbatim.

In [ ]:
process_excel_to_ttl(
    project_uri=PROJECT_URI,
    file_path=str(TEMPLATE_PATH),
    output_ttl_path=str(OUTPUT_TTL),
    uri_mode='default',
)

The function validates the output with `rdflib` itself and prints a confirmation. Let's parse it again here just to count triples and inspect the shape:

In [ ]:
import rdflib
g = rdflib.Graph()
g.parse(OUTPUT_TTL, format='turtle')
print(f'{len(g)} triples in the generated TTL')

## 9.6 Spot-check the generated TTL

Two pieces are worth eyeballing: a typical instance with its attribute IRIs, and a `dici_onto:Reference` instance with its citation triples.

In [ ]:
ttl_text = OUTPUT_TTL.read_text(encoding='utf-8')

def block_starting(needle, max_lines=20):
    lines = ttl_text.splitlines()
    for i, line in enumerate(lines):
        if needle in line:
            return '\n'.join(lines[i:i+max_lines])
    return '(not found)'

print(block_starting('EnergyConsumer/BuildingA>', max_lines=10))
print('---')
print(block_starting('Reference/swiss_energy_atlas_2024>', max_lines=8))

Notice the `BuildingA` `dici_onto:locatedIn` edge points at `Location/AlpineValley` — that's the `ClassObject` row in the spreadsheet doing its job. And `floorArea` carries a `prov:wasDerivedFrom` link because we put `swiss_energy_atlas_2024` in the `floorArea_datasource` column.

## 9.7 Upload to a fresh named graph

We'll keep this isolated from the canonical Alpine Village graph so URI shapes (`/EnergyConsumer/BuildingA` vs the hand-written `/BuildingA`) don't collide. Use `replace_existing=True` to make the cell idempotent — re-running the notebook always gives a clean graph.

In [ ]:
client.upload_ttl(
    ttl_str=ttl_text,
    graph_name=EXCEL_GRAPH,
    replace_existing=True,
)

client.sparql_api_query(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?inst ?label WHERE {{
      GRAPH {EXCEL_GRAPH} {{
        ?inst a dici_onto:EnergyConsumer ; rdfs:label ?label .
      }}
    }} ORDER BY ?inst
""", out_format='df')

Three buildings, three labels — same shape as the hand-written tutorial graph, just produced from a spreadsheet.

## 9.8 Cleanup

Drop the demo graph and remove the local TTL file so you can re-run this notebook from scratch.

In [ ]:
client.sparql_update(f"DROP GRAPH {EXCEL_GRAPH}")
OUTPUT_TTL.unlink(missing_ok=True)
print('cleaned up')

## 9.9 The same flow in the Streamlit UI

If you'd rather not write Python, the Streamlit UI has the identical path:

1. Open <http://localhost:8501> and pick the **Replica Builder** module.
2. Switch to the **Excel Import (Legacy)** tab.
3. Click *Download Excel Template* (pulls from NextCloud — see `08_nextcloud.md`) **or** drag this notebook's `sample_data/alpine_village_replica_template.xlsx` into the upload widget.
4. Pick a URI mode (the default matches what we used here) and click *Convert to TTL*.
5. The instances appear in session state and become visible in the other Replica Builder tabs (Edit, Visualize, Upload to GraphDB).

The UI calls the exact same `process_excel_to_ttl` function under the hood — there's no separate code path.

## 9.10 Bring your own template

The cells above all worked off the bundled `alpine_village_replica_template.xlsx`. The same flow works for **any** workbook you build to the [header-row convention](#9.3-Header-rows-—-what-each-one-means) above (6 or 7 rows).

Two folders are wired up out of the box:

- **Production** — `data/ingestion/input/` for source workbooks, `data/ingestion/output/` for converted TTLs. The next cell auto-detects this layout and uses it if it exists.
- **Tutorial trial** — `tutorial/sample_data/user_templates/`. Used as a fallback when the production folders aren't present. Outputs land alongside the source `.xlsx`.

Recommended workflow for your own data:

1. **Find the data folder** — printed by the next cell.
2. **Copy the seed** under a project-specific name (`my_district.xlsx`, `campus_v1.xlsx`, …). The seed is `tutorial/sample_data/alpine_village_replica_template.xlsx`. Or use the per-attribute-type CSV reference in [`data/ingestion_template/`](../data/ingestion_template/) if you'd rather paste values into a validated template you already have.
3. **Edit your copy** in Excel / LibreOffice. Add sheets for any `dici_onto:` class your project uses; keep `Reference` for citations. Don't edit the seed itself — it's the canonical example for new contributors.
4. **Run the cells below.** Your file will appear in the dropdown; pick it, set a project URI, convert.

User-supplied `.xlsx` and `.ttl` files in either folder are git-ignored — your work stays local.

In [ ]:
# Where your filled-in workbooks live, and where converted TTLs go.
# Auto-detect: prefer the production folder under data/ingestion/ if it
# exists; otherwise use the tutorial drop folder under sample_data/.
_repo_root      = pathlib.Path('..').resolve()
_prod_input     = _repo_root / 'data' / 'ingestion' / 'input'
_prod_output    = _repo_root / 'data' / 'ingestion' / 'output'
_tutorial_input = pathlib.Path('sample_data/user_templates').resolve()

if _prod_input.exists():
    USER_DIR   = _prod_input
    OUTPUT_DIR = _prod_output
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    USER_DIR   = _tutorial_input
    OUTPUT_DIR = _tutorial_input  # tutorial mode: outputs sit next to inputs

SEED = pathlib.Path('sample_data/alpine_village_replica_template.xlsx').resolve()

print('Input folder:    ', USER_DIR)
print('Output folder:   ', OUTPUT_DIR)
print('Seed template:   ', SEED)
print()
print(f'Drop your .xlsx workbooks into {USER_DIR.name}/ and re-run the next cell.')
print('On Windows: paste this path into File Explorer (Win+R → paste → Enter):')
print('  ', USER_DIR)

### Pick a template

The next cell scans `user_templates/` for `.xlsx` files and renders a dropdown. If the folder is empty it prints copy-paste instructions instead of erroring.

(`ipywidgets` ships with `jupyterlab` — if for some reason it's missing, the cell falls back to a plain text-input pattern that still works.)

In [ ]:
available = sorted(p for p in USER_DIR.glob('*.xlsx') if not p.name.startswith('~$'))

if not available:
    print('No .xlsx files found in:', USER_DIR)
    print()
    print('Copy the seed template there to get started:')
    print(f'   from:  {SEED}')
    print(f'   into:  {USER_DIR}')
    print()
    print('Or run the next cell to copy the seed automatically as my_template.xlsx.')
else:
    print(f'Found {len(available)} template(s) in {USER_DIR.name}/:')
    for p in available:
        print(f'   • {p.name}')

try:
    import ipywidgets as widgets
    from IPython.display import display

    selected = widgets.Dropdown(
        options=[(p.name, str(p)) for p in available] or [('(no files)', '')],
        description='Template:',
        layout={'width': '60%'},
    )
    project_uri_input = widgets.Text(
        value='https://example.org/my_project',
        description='Project URI:',
        layout={'width': '60%'},
    )
    uri_mode_input = widgets.Dropdown(
        options=['default', 'complete-project-uri', 'full-uri-in-cell'],
        value='default',
        description='URI mode:',
    )
    display(selected, project_uri_input, uri_mode_input)
    USE_WIDGETS = True
except ImportError:
    print()
    print('ipywidgets not installed — assign these variables manually before the next cell:')
    print("   selected_path     = '<full path to your .xlsx>'")
    print("   selected_uri      = 'https://example.org/my_project'")
    print("   selected_uri_mode = 'default'")
    USE_WIDGETS = False

### Optional: copy the seed for me

In [ ]:
import shutil

if not available:
    target = USER_DIR / 'my_template.xlsx'
    shutil.copy(SEED, target)
    print(f'copied seed → {target}')
    print('Re-run the previous cell to refresh the dropdown.')
else:
    print(f'{len(available)} template(s) already present in {USER_DIR.name}/ — skipping seed copy.')

### Convert your template

This cell reads whichever template you picked, runs `process_excel_to_ttl`, validates the output with `rdflib`, and writes the resulting TTL alongside the source `.xlsx`. Re-run safely — the output is overwritten each time.

In [ ]:
if USE_WIDGETS:
    selected_path     = selected.value
    selected_uri      = project_uri_input.value.strip()
    selected_uri_mode = uri_mode_input.value

if not selected_path:
    raise RuntimeError(f'No template selected. Drop an .xlsx in {USER_DIR} and re-run the dropdown cell.')
if not selected_uri:
    raise RuntimeError('Project URI is empty. Set it to a stable namespace for your project (e.g. https://your-org.example.org/<project>).')

src = pathlib.Path(selected_path)
dst = OUTPUT_DIR / (src.stem + '.ttl')

process_excel_to_ttl(
    project_uri=selected_uri,
    file_path=str(src),
    output_ttl_path=str(dst),
    uri_mode=selected_uri_mode,
)

import rdflib
g = rdflib.Graph()
g.parse(dst, format='turtle')
print()
print(f'wrote {len(g)} triples → {dst}')

### Upload to a per-template named graph

Optional: push the result into GraphDB under a graph URI derived from the template filename. The notebook keeps your conversion isolated from the bundled Alpine Village graph and from any previous run.

In [ ]:
user_graph = f"<{selected_uri}>"

client.upload_ttl(
    ttl_str=dst.read_text(encoding='utf-8'),
    graph_name=user_graph,
    replace_existing=True,
)

client.sparql_api_query(f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?inst ?type ?label WHERE {{
      GRAPH {user_graph} {{
        ?inst a ?type .
        OPTIONAL {{ ?inst rdfs:label ?label }}
        FILTER NOT EXISTS {{ ?inst a ?_attrType . FILTER(STRSTARTS(STR(?_attrType), 'https://digicities.info/ontology#') && STRENDS(STR(?_attrType), 'Attribute')) }}
      }}
    }} ORDER BY ?type ?inst LIMIT 50
""", out_format='df')

That's the whole loop: drop a workbook in `data/ingestion/input/` (or `tutorial/sample_data/user_templates/` for tutorial trials), refresh the dropdown, convert, upload. The generated TTL writes to `data/ingestion/output/` when working from the production folder, or alongside the source `.xlsx` when working from the tutorial folder. The same `process_excel_to_ttl` function the Streamlit *Excel Import* tab calls is doing the work — no separate code path.

## What you just did

- Read the **6/7-row header convention** every Replica Builder Excel template uses, including the optional `LinkedClassObjectType` row that overrides URI generation for `ClassObject` attributes.
- Mapped each of the 15 supported attribute types to the header cells it reads.
- Converted a populated workbook into TTL with `process_excel_to_ttl`.
- Validated the output with `rdflib` and uploaded it to a dedicated named graph.
- Saw how citations from the `Reference` sheet flow into `prov:wasDerivedFrom` triples.
- Ran the **Bring-your-own-template** flow against either `data/ingestion/input/` (production) or `tutorial/sample_data/user_templates/` (tutorial fallback), with a per-attribute-type CSV reference at `data/ingestion_template/` for paste-into-your-validated-template work.

For larger projects this is usually the **fastest** way to bootstrap a digital twin: hand the template (or the CSV reference) to a domain expert, get back a populated workbook, and ingest it in one shot.